# task 3 -> test-time compute: sampling, selection and adaptive inference

## imports

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#the scripts use paths relative to the repo root (configs/, data/, results/), so the notebook runs from there
if not Path('configs').exists():
    os.chdir('..')

from common.utils import load_yaml, read_jsonl, repo_path

## global env variables ->

In [ ]:
#papermill overrides these for the smoke test, the defaults are the real run
CONFIG = 'configs/task3.yaml'
RUN_SCRIPTS = True #False -> only rebuild the tables and figures from results already on disk

In [ ]:
cfg = load_yaml(CONFIG)
RESULTS = repo_path(cfg['results_dir'])
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
DATA = repo_path(cfg['data_dir'])
#the exact configuration of this run (smoke overrides included), kept with its results
import yaml
(RESULTS / 'run_config.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))

DATASETS = list(cfg['datasets']) #math, mathvista
BUDGETS = cfg['budgets']
NAMES = {'math': 'MATH (Qwen2.5-3B)', 'mathvista': 'MathVista (Qwen2.5-VL-3B)'}
#one fixed colour per method, the same in every figure
COLORS = {'first': '#52514e', 'self_consistency': '#2a78d6', 'mean_logp': '#9b9a95', 'self_certainty': '#eb6834',
          'self_certainty_rwc': '#b8441a', 'shortest': '#c9b458', 'vision': '#7fb77e', 'vision_rwc': '#3d8b3c',
          'visconf': '#b48ad6', 'visconf_rwc': '#7a3fb0', 'visconf_token': '#e3a3c6', 'visconf_token_rwc': '#b0507f'}

PYTHON = sys.executable #the interpreter running this notebook, so the scripts use the same environment

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.axisbelow': True, 'lines.linewidth': 2})


def run(command):
    #runs one script as its own process, exactly as it would be typed in a terminal. each script writes its results
    #to disk, nothing is passed through notebook state. finished work is skipped
    if RUN_SCRIPTS:
        get_ipython().system(command)
        assert get_ipython().user_ns['_exit_code'] == 0, f'failed -> {command}'


def load_result(name):
    return json.loads((RESULTS / name).read_text())


def eval_only(frame):
    #the reported numbers are on the 800 held-out questions. development numbers stay in the csv files
    return frame[frame['split'] == 'eval']

## 1. data -> join and check the supplied tables, splits, answer groups and correctness
cpu only. downloads ATML-PA3-Data and the mathvista metadata once, verifies token-list alignment, makes the
200 / 800 split (and the 50 live questions), and labels every rollout.

In [ ]:
run(f'{PYTHON} -m task3_ttc.prepare_data --config {CONFIG}')
prep = load_result('prepare_summary.json')
print('scorer ->', prep['scorer'])
for d in DATASETS:
    info = prep['datasets'][d]
    assert all(v is True or v == 0 for v in info['checks'].values()), info['checks']
    print(f"{d}: every check passed -> {sorted(info['checks'])}")
display(pd.DataFrame({d: {k: v for k, v in prep['datasets'][d].items() if k != 'checks'} for d in DATASETS}).astype(object))

## 2. offline selection and signals (steps 1 to 3)
ten seeded orders per question, budgets N = 1 to 32 as nested prefixes, every selection rule on the same candidates.

In [ ]:
run(f'{PYTHON} -m task3_ttc.offline --config {CONFIG}')
summary = pd.read_csv(RESULTS / 'offline_summary.csv')
boot = pd.read_csv(RESULTS / 'offline_bootstrap.csv')
summary_eval = eval_only(summary)

## 3. adaptive sampling (step 5), settings chosen on the development split, then frozen

In [ ]:
run(f'{PYTHON} -m task3_ttc.adaptive --config {CONFIG}')

## 4. live generation (step 4), the only gpu step

In [ ]:
run(f'{PYTHON} -m task3_ttc.live --config {CONFIG}')

## step 1 -> opportunity from repeated sampling
firstans accuracy, oracle coverage, distinct answer groups (NO_ANSWER counts as a group), and the decomposition of
self-consistency failures into not generated (no correct candidate) and missed (a correct candidate was available).

In [ ]:
rows = []
for d in DATASETS:
    s = summary_eval[summary_eval['dataset'] == d].set_index(['N', 'method'])
    for n in BUDGETS:
        sc = s.loc[(n, 'self_consistency')]
        rows.append({'dataset': d, 'N': n, 'firstans_accuracy': s.loc[(n, 'first'), 'accuracy'], 'oracle_coverage': sc['oracle'],
                     'distinct_groups': sc['n_groups'], 'sc_accuracy': sc['accuracy'], 'sc_missed': sc['missed'],
                     'not_generated': sc['not_generated'], 'tokens_all_candidates': sc['cost'],
                     'tokens_firstans': s.loc[(n, 'first'), 'cost']})
step1 = pd.DataFrame(rows)
step1.to_csv(RESULTS / 'table_step1_sampling.csv', index=False)
display(step1.round(4))

In [ ]:
#accuracy and coverage against candidates (left) and generated tokens (right). dashed = oracle coverage
main_methods = ['first', 'self_consistency', 'self_certainty', 'self_certainty_rwc']
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
for row, d in enumerate(DATASETS):
    s = summary_eval[summary_eval['dataset'] == d]
    methods = main_methods + (['visconf', 'visconf_rwc'] if d == 'mathvista' else [])
    for col, x in enumerate(['N', 'cost']):
        ax = axes[row, col]
        for m in methods:
            t = s[s['method'] == m]
            ax.plot(t[x], t['accuracy'], marker='o', ms=4, color=COLORS[m], label=m)
        o = s[s['method'] == 'self_consistency']
        ax.plot(o[x], o['oracle'], ls='--', color='#1a1a19', label='oracle coverage')
        ax.set_xscale('log', base=2 if x == 'N' else 10)
        ax.set_xlabel('candidates N' if x == 'N' else 'mean generated tokens per question')
        ax.set_ylabel('accuracy (held-out split)')
        ax.set_title(NAMES[d])
    axes[row, 0].set_xticks(BUDGETS, [str(n) for n in BUDGETS])
axes[0, 1].legend(fontsize=8, frameon=False, loc='lower right')
axes[1, 1].legend(fontsize=8, frameon=False, loc='lower right')
fig.tight_layout()
fig.savefig(FIGURES / 'accuracy_coverage.png', dpi=200)
plt.show()

In [ ]:
#failure decomposition of self-consistency per budget -> correct, missed (correct candidate available), not generated
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, d in zip(axes, DATASETS):
    t = step1[step1['dataset'] == d]
    x = np.arange(len(t))
    ax.bar(x, t['sc_accuracy'], color='#2a78d6', label='selected correctly')
    ax.bar(x, t['sc_missed'], bottom=t['sc_accuracy'], color='#eb6834', label='correct available, missed')
    ax.bar(x, t['not_generated'], bottom=t['sc_accuracy'] + t['sc_missed'], color='#cfcdc6', label='no correct candidate')
    ax.set_xticks(x, [str(n) for n in t['N']])
    ax.set_xlabel('candidates N')
    ax.set_ylabel('fraction of questions')
    ax.set_title(NAMES[d])
axes[1].legend(fontsize=8, frameon=False, loc='lower left')
fig.tight_layout()
fig.savefig(FIGURES / 'failure_decomposition.png', dpi=200)
plt.show()

## step 2 -> quality scoring and answer aggregation
accuracy of every rule at every budget (held-out split), then paired bootstrap differences (1,000 question resamples).

In [ ]:
for d in DATASETS:
    t = summary_eval[summary_eval['dataset'] == d].pivot(index='method', columns='N', values='accuracy')
    t = t.loc[[m for m in COLORS if m in t.index]]
    t.to_csv(RESULTS / f'table_methods_{d}.csv')
    print(NAMES[d])
    display(t.round(4))

In [ ]:
#main differences at N = 32 (and the full set for every N in offline_bootstrap.csv)
b = eval_only(boot)
b32 = b[b['N'] == 32].copy()
b32['significant'] = (b32['ci_low'] > 0) | (b32['ci_high'] < 0)
b32.to_csv(RESULTS / 'table_bootstrap_n32.csv', index=False)
display(b32.drop(columns=['split', 'N']).round(4))

## step 3 -> what the signals capture (N = 32)
within-question associations. quartile gap = correctness of the top 8 minus the bottom 8 rollouts by the score,
averaged over questions (mixed = questions with both correct and incorrect rollouts, the only ones where it can
differ from zero). shortest-rollout selection is in the method tables above as the length-only control.

In [ ]:
signals = load_result('signals_summary.json')
rows = []
for d in DATASETS:
    s = signals[d]['eval']
    for name, v in s['scores'].items():
        rows.append({'dataset': d, 'score': name, **v})
    print(f"{d}: {s['mixed_questions']} of {s['questions']} questions mixed, mean length correct minus incorrect "
          f"(mixed) = {s['length_correct_minus_incorrect_mixed']:.1f} tokens")
sig_table = pd.DataFrame(rows)
sig_table.to_csv(RESULTS / 'table_signals.csv', index=False)
display(sig_table.round(4))

In [ ]:
#mathvista -> confidence-vision correlation within each rollout (pearson over its tokens), averaged per question,
#then summarized across questions. corr_kl_v uses the raw token kl, corr_c_v the calibrated c (all 32 rollouts),
#cov_c_v = Z^token - Z
cv = signals['mathvista']['eval']['confidence_vision']
display(pd.DataFrame({k: v for k, v in cv.items() if isinstance(v, dict)}).round(4))
print({k: v for k, v in cv.items() if not isinstance(v, dict)})

In [ ]:
#tokenwise fusion vs trajectory-level visconf under both aggregation rules, every budget
s = summary_eval[summary_eval['dataset'] == 'mathvista'].pivot(index='N', columns='method', values='accuracy')
fusion = s[['visconf', 'visconf_token', 'visconf_rwc', 'visconf_token_rwc']].copy()
bb = b[(b['dataset'] == 'mathvista') & (((b['method'] == 'visconf_token') & (b['baseline'] == 'visconf')) |
                                       ((b['method'] == 'visconf_token_rwc') & (b['baseline'] == 'visconf_rwc')))]
for m in ['visconf_token', 'visconf_token_rwc']:
    t = bb[bb['method'] == m].set_index('N')
    fusion[f'{m}_minus_trajectory'] = t['difference']
    fusion[f'{m}_ci'] = t.apply(lambda r: f"[{r['ci_low']:.4f}, {r['ci_high']:.4f}]", axis=1)
fusion.to_csv(RESULTS / 'table_fusion.csv')
display(fusion.round(4))

In [ ]:
#illustrative trajectories (fixed rule in offline.py -> first 3 mixed evaluation questions per dataset, lowest-id
#correct and incorrect rollout). token signals averaged over 50 equal slices of the generation
traj = load_result('trajectories.json')
series = {'math': ['kl', 'logp'], 'mathvista': ['kl', 'c', 'v']}
for d in DATASETS:
    ts = [t for t in traj if t['dataset'] == d]
    ids = list(dict.fromkeys(t['sample_id'] for t in ts))
    fig, axes = plt.subplots(len(series[d]), len(ids), figsize=(4 * len(ids), 2.4 * len(series[d])), squeeze=False)
    for j, s_id in enumerate(ids):
        for i, key in enumerate(series[d]):
            ax = axes[i, j]
            for t in [t for t in ts if t['sample_id'] == s_id]:
                y = [np.nan if v is None else v for v in t['binned'][key]]
                ax.plot(np.linspace(0, 1, len(y)), y, color='#2a78d6' if t['correct'] else '#eb6834',
                        label=f"{'correct' if t['correct'] else 'incorrect'} ({t['n_tokens']} tok, ans {t['answer'][:12]})")
            ax.set_ylabel(key)
            if i == 0:
                ax.set_title(f'{d} {s_id}', fontsize=9)
                ax.legend(fontsize=7, frameon=False)
            if i == len(series[d]) - 1:
                ax.set_xlabel('relative position in the generation')
    fig.tight_layout()
    fig.savefig(FIGURES / f'trajectories_{d}.png', dpi=200)
    plt.show()

## step 4 -> parallel sampling vs sequential refinement (live, 50 math questions)
the refinement cost includes the shared initial response and the longer revision prompt.

In [ ]:
live = load_result('live_summary.json')
live_table = pd.DataFrame(live['strategies']).T
live_table.to_csv(RESULTS / 'table_live.csv')
display(live_table.round(4))
print('refinement transitions (relative to the shared initial answer) ->', live['refinement_transitions'])
display(pd.DataFrame(live['bootstrap']).round(4))
print(f"gpu {live['gpu']}, {live['dtype']}, batch size {live['batch_size']}")

## step 5 -> adaptive sampling
development grid, the frozen settings, then held-out accuracy, cost and stopping distribution next to fixed-N
self-consistency.

In [ ]:
grid = pd.read_csv(RESULTS / 'adaptive_dev_grid.csv')
selected = load_result('adaptive_selected.json')
for d in DATASETS:
    g = grid[grid['dataset'] == d]
    print(NAMES[d], '-> development accuracy (rows tau, columns gamma; agreement = no gamma)')
    q = g[g['family'] == 'quality'].pivot(index='tau', columns='gamma', values='accuracy')
    q.insert(0, 'agreement', g[g['family'] == 'agreement'].set_index('tau')['accuracy'])
    display(q.round(4))
    print('fixed 32 ->', g[g['family'] == 'fixed_32'][['accuracy', 'cost']].round(4).to_dict('records'))
    display(pd.DataFrame(selected[d]).T)

In [ ]:
adaptive = pd.read_csv(RESULTS / 'adaptive_eval.csv')
display(adaptive.round(4))
ab = pd.read_csv(RESULTS / 'adaptive_bootstrap.csv')
display(ab.round(4))
early = pd.read_csv(RESULTS / 'adaptive_early_stops.csv')
orders = cfg['orders']
counts = early.groupby(['dataset', 'policy']).agg(incorrect_early_stops=('order', 'size'),
                                                  recovered_by_fixed_32=('recovered_by_fixed_32', 'sum'),
                                                  questions=('sample_id', 'nunique'))
counts['per_question_order_rate'] = counts['incorrect_early_stops'] / (800 * orders) if not cfg.get('question_limit') else np.nan
display(counts)

## step 6 -> compute allocation and failure cases
accuracy against generated tokens with the adaptive policies placed at their observed cost, stopping behaviour
against agreement and the (retrospective) fraction of correct candidates, and whether more samples add new answers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, d in zip(axes, DATASETS):
    f = adaptive[(adaptive['dataset'] == d) & adaptive['policy'].str.startswith('fixed')]
    ax.plot(f['cost'], f['accuracy'], marker='o', color=COLORS['self_consistency'], label='fixed-N self-consistency')
    for _, r in f.iterrows():
        ax.annotate(r['policy'].split('_')[1], (r['cost'], r['accuracy']), fontsize=7, xytext=(3, -9), textcoords='offset points')
    o = summary_eval[(summary_eval['dataset'] == d) & (summary_eval['method'] == 'self_consistency')]
    ax.plot(o['cost'], o['oracle'], ls='--', color='#1a1a19', label='oracle coverage')
    for policy, marker in [('agreement', 's'), ('quality', 'D')]:
        a = adaptive[(adaptive['dataset'] == d) & (adaptive['policy'] == policy)]
        ax.scatter(a['cost'], a['accuracy'], marker=marker, s=60, zorder=3, label=f'adaptive {policy}')
    ax.set_xscale('log')
    ax.set_xlabel('mean generated tokens per question')
    ax.set_ylabel('accuracy (held-out)')
    ax.set_title(NAMES[d])
axes[0].legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / 'adaptive_cost.png', dpi=200)
plt.show()

In [ ]:
#stopping vs the full pool. agreement = plurality share of the 32 rollouts, correct fraction = share of correct
#rollouts among the 32 (retrospective only, never used by a controller)
per_q = pd.read_csv(RESULTS / 'adaptive_per_question.csv')
rows = []
for d in DATASETS:
    groups = load_result(f'answer_groups_{d}.json')
    full = pd.DataFrame([{'sample_id': s, 'plurality_share': max(g['size'] for g in v['groups']) / 32,
                          'correct_fraction': sum(g['size'] for g in v['groups'] if g['correct']) / 32} for s, v in groups.items()])
    t = per_q[per_q['dataset'] == d].merge(full, on='sample_id')
    for col in ['correct_fraction', 'plurality_share']:
        t[f'{col}_bin'] = pd.cut(t[col], [-0.01, 0.25, 0.5, 0.75, 0.999, 1.0], labels=['0 to .25', '.25 to .5', '.5 to .75', '.75 to <1', '1'])
        g = t.groupby(['policy', f'{col}_bin'], observed=True).agg(questions=('sample_id', 'size'), mean_rollouts=('rollouts', 'mean'),
                                                                  accuracy=('accuracy', 'mean')).reset_index()
        g.insert(0, 'dataset', d)
        g = g.rename(columns={f'{col}_bin': 'bin'})
        g.insert(2, 'by', col)
        rows.append(g)
stop_table = pd.concat(rows)
stop_table.to_csv(RESULTS / 'table_stopping_vs_pool.csv', index=False)
display(stop_table.round(3))

In [ ]:
#do extra samples add alternatives or repeat answers? new groups per doubling and the coverage each doubling adds
rows = []
for d in DATASETS:
    t = step1[step1['dataset'] == d].reset_index(drop=True)
    for i in range(1, len(t)):
        rows.append({'dataset': d, 'from_N': t['N'][i - 1], 'to_N': t['N'][i],
                     'new_groups_per_added_rollout': (t['distinct_groups'][i] - t['distinct_groups'][i - 1]) / (t['N'][i] - t['N'][i - 1]),
                     'coverage_gain': t['oracle_coverage'][i] - t['oracle_coverage'][i - 1],
                     'sc_accuracy_gain': t['sc_accuracy'][i] - t['sc_accuracy'][i - 1]})
growth = pd.DataFrame(rows)
growth.to_csv(RESULTS / 'table_sample_growth.csv', index=False)
display(growth.round(4))

In [ ]:
#representative cases, chosen by fixed rules (no hand picking) ->
#  selection disagreement  first 3 held-out questions (saved permutation order) per dataset where, on order 0 at N = 32,
#                          self-consistency and the quality rule (self-certainty for math, visconf for mathvista)
#                          select different answers and exactly one of them is correct
#  revision                for each refinement transition that occurs, the first live question (live order) showing it
#  premature stopping      first 3 held-out questions per dataset where the quality-aware controller stopped early
#                          on order 0 with a wrong answer that fixed-32 self-consistency gets right
splits = load_result('splits.json')
examples = {'selection_disagreement': [], 'revision': [], 'premature_stopping': []}
for d in DATASETS:
    groups = load_result(f'answer_groups_{d}.json')
    text = pd.read_parquet(DATA / f'rollouts_{d}.parquet', columns=['sample_id', 'rollout_id', 'question_text'])
    text = text[text['rollout_id'] == 0].set_index('sample_id')['question_text']
    sel = pd.read_parquet(RESULTS / f'selections_{d}.parquet')
    sel = sel[(sel['order'] == 0) & (sel['N'] == 32)].set_index('sample_id')
    rule = cfg['adaptive_quality_score'][d]
    key = lambda s, g: groups[s]['groups'][int(g)]['key']
    for s in splits[d]['eval']:
        if s in sel.index and len(examples['selection_disagreement']) < 3 * (DATASETS.index(d) + 1):
            r = sel.loc[s]
            if r['self_consistency'] != r[rule] and r['self_consistency_correct'] != r[f'{rule}_correct']:
                examples['selection_disagreement'].append({
                    'dataset': d, 'sample_id': s, 'question': text[s][:300], 'gold': groups[s]['gold'],
                    'self_consistency': key(s, r['self_consistency']), 'sc_correct': bool(r['self_consistency_correct']),
                    rule: key(s, r[rule]), f'{rule}_correct': bool(r[f'{rule}_correct']),
                    'group_sizes_32': {g['key']: g['size'] for g in groups[s]['groups']}})
    e = early[(early['dataset'] == d) & (early['policy'] == 'quality') & (early['order'] == 0) & early['recovered_by_fixed_32']]
    for s in [s for s in splits[d]['eval'] if s in set(e['sample_id'])][:3]:
        r = e[e['sample_id'] == s].iloc[0]
        examples['premature_stopping'].append({'dataset': d, 'sample_id': s, 'question': text[s][:300], 'gold': groups[s]['gold'],
                                               'stop_n': int(r['stop_n']), 'stopped_answer': key(s, r['stopped_answer_group']),
                                               'fixed_32_answer': key(s, r['fixed_32_answer_group'])})
lq = pd.read_csv(RESULTS / 'live_questions.csv')
ref = lq[lq['strategy'] == 'refinement']
for name, mask in {'wrong_to_correct': ~ref['initial_correct'].astype(bool) & ref['correct'],
                   'correct_to_wrong': ref['initial_correct'].astype(bool) & ~ref['correct'],
                   'changed_wrong_to_wrong': ~ref['initial_correct'].astype(bool) & ~ref['correct'] & (ref['initial_answer'] != ref['answer'])}.items():
    if mask.any():
        r = ref[mask].iloc[0]
        examples['revision'].append({'transition': name, 'sample_id': r['sample_id'], 'initial_answer': r['initial_answer'],
                                     'revised_answer': r['answer']})
(RESULTS / 'examples.json').write_text(json.dumps(examples, indent=2, ensure_ascii=False))
for k, v in examples.items():
    print(k)
    display(pd.DataFrame(v))

## resources

In [ ]:
rows = [{'step': 'data preparation (cpu)', 'minutes': prep['seconds'] / 60},
        {'step': 'offline selection + signals (cpu)', 'minutes': load_result('offline_run.json')['seconds'] / 60},
        {'step': 'adaptive simulation (cpu)', 'minutes': load_result('adaptive_run.json')['seconds'] / 60}]
for strategy, s in live['strategies'].items():
    rows.append({'step': f'live -> {strategy} (summed per-call time, initial response included for refinement)',
                 'minutes': s['total_seconds'] / 60, 'peak_memory_gb': s['peak_memory_gb']})
resources = pd.DataFrame(rows)
resources.to_csv(RESULTS / 'resources.csv', index=False)
display(resources.round(3))
print(f"live generation on {live['gpu']}; offline token costs exclude prefill, attention collection and scoring")